In [18]:
import os
import pandas as pd
import numpy as np

In [26]:
import os
import pandas as pd
import numpy as np

# Define the directory containing the .txt files
directory = 'project_folder/Wind/Extreme wind shear and gust/NOAA Buoy data'

# Define the headers and units
headers = ["YYYY (year)", "MM (month)", "DD (day)", "hh (hour)", "mm (minute)", "WDIR (degT)", "WSPD (m/s)", "GST (m/s)", 
           "WVHT (m)", "DPD (sec)", "APD (sec)", "MWD (deg)", "PRES (hPa)", "ATMP (degC)", "WTMP (degC)", "DEWP (degC)", 
           "VIS (nmi)", "TIDE (ft)"]

# Initialize an empty list to store DataFrames
dataframes = []

# Get the list of files and sort them
files = sorted([f for f in os.listdir(directory) if f.endswith('.txt')])

# Loop through all sorted files in the directory
for filename in files:
    filepath = os.path.join(directory, filename)
    
    try:
        # Extract year from filename (e.g., 44009h2007.txt -> 2007)
        filename_year = None
        if 'h' in filename and filename.endswith('.txt'):
            try:
                year_part = filename.split('h')[1].split('.')[0]
                if len(year_part) == 4 and year_part.isdigit():
                    filename_year = int(year_part)
            except:
                pass
        
        # Read the first lines to determine the file format
        with open(filepath, 'r') as file:
            header_lines = []
            for i in range(5):  # Read up to 5 lines to find headers
                line = file.readline().strip()
                if line:
                    header_lines.append(line)
                if len(header_lines) >= 2:
                    break
                    
        if len(header_lines) < 1:
            print(f"Skipping {filename}: Not enough header lines")
            continue
            
        first_line = header_lines[0]
        second_line = header_lines[1] if len(header_lines) > 1 else ""
        
        # Determine the format of the file based on the header
        if '#YY' in first_line and '#yr' in second_line:  # Post-2007 with units in second line
            year_format = 'double_with_header'
            skiprows = 2
        elif ('YY' in first_line) and ('yr' not in second_line):  # Pre-2007 format with YY
            year_format = 'double'
            skiprows = 1  # Only skip the header line
        elif 'YYYY' in first_line and 'TIDE' not in first_line:
            year_format = 'four'
            skiprows = 1  # Only skip the header line
        elif 'YYYY' in first_line and 'TIDE' in first_line and 'mm' not in first_line:
            year_format = 'four_tide'
            skiprows = 1  # Only skip the header line
        elif 'YYYY' in first_line and 'mm' in first_line:  # Newer format with units in second line
            year_format = 'full'
            skiprows = 2 if any(unit in second_line for unit in ['yr', 'mo', 'degT']) else 1
        else:
            # Default case
            year_format = 'unknown'
            skiprows = 1
        
        print(f"Processing {filename}: Format '{year_format}', skipping {skiprows} rows")
        
        # Read the data
        df = pd.read_csv(filepath, sep='\s+', skiprows=skiprows, header=None, on_bad_lines='skip')
        
        # Skip empty dataframes
        if df.empty:
            print(f"Skipping {filename}: Empty data")
            continue
            
        # Use filename_year to validate/correct the year data
        first_year_in_file = df[0].iloc[0] if len(df) > 0 else None
        
        # Fix years based on the file format and filename
        if first_year_in_file is not None:
            if len(str(int(first_year_in_file))) <= 2:  # Two-digit year
                df[0] = df[0].apply(lambda x: 1900 + int(x) if int(x) >= 80 else 2000 + int(x))
            elif filename_year is not None and abs(int(first_year_in_file) - filename_year) > 100:
                # If the year in the file is very different from the filename year
                print(f"Correcting year in {filename}: from {first_year_in_file} to {filename_year}")
                df[0] = filename_year
        
        # Set 'mm' to '0' if missing
        if 'mm' not in first_line:
            df.insert(4, 'mm', 0)
        
        # Ensure 'PRES' (or 'BAR') and 'TIDE' are present
        pres_col_index = 12
        if 'PRES' not in first_line and 'BAR' in first_line:
            # BAR is in the same position as PRES
            pass
        elif 'PRES' not in first_line and 'BAR' not in first_line:
            df.insert(pres_col_index, 'PRES', None)
            
        if 'TIDE' not in first_line:
            df['TIDE'] = None
        
        # Rename 'WD' to 'WDIR' if present
        if 'WD' in first_line and 'WDIR' not in first_line:
            col_index = 5
            df.rename(columns={df.columns[col_index]: 'WDIR'}, inplace=True)
        
        # Adjust the number of columns to match the headers
        if df.shape[1] > len(headers):
            df = df.iloc[:, :len(headers)]
        elif df.shape[1] < len(headers):
            for _ in range(len(headers) - df.shape[1]):
                df[len(df.columns)] = None
        
        # Assign headers
        df.columns = headers
        
        # Ensure all years are within reasonable range
        df["YYYY (year)"] = df["YYYY (year)"].apply(
            lambda x: int(x) if isinstance(x, (int, float)) and 1984 <= int(x) <= 2025 else filename_year
        )
        
        # Append the DataFrame to the list
        dataframes.append(df)
        print(f"Processed {filename} with {len(df)} rows, year range: {df['YYYY (year)'].min()}-{df['YYYY (year)'].max()}")
        
    except Exception as e:
        print(f"Error processing {filename}: {str(e)}")
        continue

# Combine all DataFrames into a single DataFrame
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    
    # Ensure YYYY column is always integer
    combined_df["YYYY (year)"] = pd.to_numeric(combined_df["YYYY (year)"], errors='coerce').fillna(2000).astype(int)
    
    # Sort the dataframe by year, month, day, hour, minute
    combined_df = combined_df.sort_values(by=["YYYY (year)", "MM (month)", "DD (day)", "hh (hour)", "mm (minute)"])
    
    # Reset the index after sorting
    combined_df = combined_df.reset_index(drop=True)
    
    # Display the combined DataFrame summary
    print("\nCombined DataFrame:")
    print(f"Total rows: {len(combined_df)}")
    print(f"Year range: {combined_df['YYYY (year)'].min()} - {combined_df['YYYY (year)'].max()}")
else:
    print("No valid data found in the directory")

Processing 44009h1984.txt: Format 'double', skipping 1 rows
Processed 44009h1984.txt with 8411 rows, year range: 1984-1984
Processing 44009h1985.txt: Format 'double', skipping 1 rows
Processed 44009h1985.txt with 7490 rows, year range: 1985-1985
Processing 44009h1986.txt: Format 'double', skipping 1 rows
Processed 44009h1986.txt with 5504 rows, year range: 1986-1986
Processing 44009h1987.txt: Format 'double', skipping 1 rows
Processed 44009h1987.txt with 8683 rows, year range: 1987-1987
Processing 44009h1988.txt: Format 'double', skipping 1 rows
Processed 44009h1988.txt with 6471 rows, year range: 1988-1988
Processing 44009h1989.txt: Format 'double', skipping 1 rows
Processed 44009h1989.txt with 8425 rows, year range: 1989-1989
Processing 44009h1990.txt: Format 'double', skipping 1 rows
Processed 44009h1990.txt with 8548 rows, year range: 1990-1990
Processing 44009h1991.txt: Format 'double', skipping 1 rows
Processed 44009h1991.txt with 8725 rows, year range: 1991-1991
Processing 44009

/var/folders/c9/zx22n7c1713_ck39lvd8nlx00000gn/T/ipykernel_13540/914263355.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(dataframes, ignore_index=True)


In [27]:
combined_df

,YYYY (year),MM (month),DD (day),hh (hour),mm (minute),WDIR (degT),WSPD (m/s),GST (m/s),WVHT (m),DPD (sec),APD (sec),MWD (deg),PRES (hPa),ATMP (degC),WTMP (degC),DEWP (degC),VIS (nmi),TIDE (ft)
0,1984,1,4,2,0,230,2.0,3.0,99.00,99.00,99.00,999,1025.1,2.0,7.1,999.0,99.0,NaN
1,1984,1,4,3,0,210,2.0,4.0,99.00,99.00,99.00,999,1024.8,2.3,7.1,999.0,99.0,NaN
2,1984,1,4,4,0,220,3.0,5.0,99.00,99.00,99.00,999,1024.0,2.7,7.0,999.0,99.0,NaN
3,1984,1,4,5,0,210,5.0,6.0,99.00,99.00,99.00,999,1023.6,2.8,7.0,999.0,99.0,NaN
4,1984,1,4,6,0,210,5.0,7.0,99.00,99.00,99.00,999,1022.9,3.1,7.0,999.0,99.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490008,2025,2,28,23,10,172,1.8,2.6,0.87,8.33,6.08,136,1009.2,7.1,6.2,4.6,99.0,99.0
490009,2025,2,28,23,20,169,1.5,2.0,99.00,99.00,99.00,999,1009.3,7.1,5.9,4.6,99.0,99.0
490010,2025,2,28,23,30,170,2.1,2.5,99.00,99.00,99.00,999,1009.4,7.0,6.1,4.7,99.0,99.0
490011,2025,2,28,23,40,181,2.5,3.2,0.75,7.14,5.58,168,1009.3,7.0,6.2,4.8,99.0,99.0
